# GRACE-DF: Baseline Models Training & Table 1 Generation

**Project:** Grounded, Robust And Calibrated Evidence for Deepfake Detection (GRACE-DF)  
**Target:** IEEE Transactions on Information Forensics and Security (TIFS)  
**Objective:** 
1. Train the five Shakya et al. (IEEE ISDFS 2026) baselines: ResNet-18, MobileNetV3-S, EfficientNet-B0, ConvNeXt-T, ViT-B/16.
2. Train ARC-Net (PLOS ONE 2026: EfficientNet-B0 + Residual Attention).
3. Repeat across 3 random seeds (42, 43, 44) with full RNG preservation.
4. Adhere strictly to **Kaggle Rule 1** (auto-resume from `ckpt_last.pt`) and **Kaggle Rule 2** (`run.json` logging).
5. Auto-generate **Table 1** (clean baseline reproduction) in JSON, Markdown, and publication-ready LaTeX.

In [ ]:
# GPU and environment verification
import os
import sys
import shutil
from pathlib import Path
import torch

print(f"PyTorch: {torch.__version__}")
assert torch.cuda.is_available(), "CUDA GPU is required for training! Enable GPU in Kaggle settings."
device_name = torch.cuda.get_device_name(0)
print(f"Active GPU: {device_name}")

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
RUNS_DIR = WORKING_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Ensure project root is in sys.path
REPO_ROOT = Path("/kaggle/working/grace-df") if Path("/kaggle/working/grace-df").exists() else Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# Locate WebDataset shards from Kaggle Dataset
INPUT_DIR = Path("/kaggle/input")
possible_shard_dirs = list(INPUT_DIR.glob("**/shards_train_384px")) + list(INPUT_DIR.glob("**/train_384px")) + list(WORKING_DIR.glob("**/shards_train_384px"))

if possible_shard_dirs:
    shard_dir = possible_shard_dirs[0]
    print(f"Found training shards in: {shard_dir}")
    shard_paths = sorted(list(shard_dir.glob("*.tar")))
    print(f"Total shard archives: {len(shard_paths)}")
else:
    print("Training shards not yet found in /kaggle/input. Using local fallback or synthetic test data.")
    shard_paths = []

In [ ]:
# Import GRACE-DF training and evaluation modules
from src.models import build_model
from src.train.baseline import BaselineTrainingConfig, train_baseline
from src.data.shards import ShardDataset
from src.eval.evaluate_drift import evaluate_model_drift

# Architectures to train
ARCHITECTURES = ["resnet18", "mobilenetv3_s", "effnet_b0", "convnext_t", "vit_b_16", "arcnet"]
SEEDS = [42, 43, 44]

print(f"Configured {len(ARCHITECTURES)} architectures across {len(SEEDS)} seeds ({len(ARCHITECTURES)*len(SEEDS)} total runs).")

In [ ]:
# Execute training loop with Rule 1 atomic checkpoint resume
for arch in ARCHITECTURES:
    for seed in SEEDS:
        run_name = f"baseline_{arch}_seed{seed}"
        print(f"\n{'='*30} Starting {run_name} {'='*30}")
        
        cfg = BaselineTrainingConfig(
            arch=arch,
            run_name=run_name,
            seed=seed,
            epochs=8,
            lr=1e-4 if arch != "vit_b_16" else 3e-5,
            batch_size=32 if arch != "vit_b_16" else 16,
            grad_accum_steps=1 if arch != "vit_b_16" else 2,
            amp=True,
            device="cuda" if torch.cuda.is_available() else "cpu",
            runs_dir=str(RUNS_DIR)
        )
        
        # In the presence of real shards:
        if shard_paths:
            n_val = max(1, len(shard_paths) // 10)
            train_shards = shard_paths[:-n_val]
            val_shards = shard_paths[-n_val:]
            train_ds = ShardDataset(train_shards, target_size=(224, 224))
            val_ds = ShardDataset(val_shards, target_size=(224, 224))
            from torch.utils.data import DataLoader
            train_loader = DataLoader(train_ds, batch_size=cfg.batch_size)
            val_loader = DataLoader(val_ds, batch_size=cfg.batch_size)
            
            train_summary = train_baseline(cfg, train_loader, val_loader)
            print(f"Run {run_name} finished: Val Acc = {train_summary['val_metrics']['val_acc']:.4f}")
        else:
            print(f"Skipping {run_name} execution — awaiting Kaggle dataset mount.")

In [ ]:
# Auto-generate Table 1 from run.json outputs
from scripts.generate_table1 import collect_baseline_metrics, generate_latex, generate_markdown

table_data = collect_baseline_metrics(RUNS_DIR)
md_report = generate_markdown(table_data)
latex_code = generate_latex(table_data)

print("=== Generated Table 1 (Markdown) ===")
print(md_report)

with open(RUNS_DIR / "table1_clean_baselines.md", "w") as f:
    f.write(md_report)
with open(RUNS_DIR / "table1_clean_baselines.tex", "w") as f:
    f.write(latex_code)

print("Table 1 exported successfully!")